In [1]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms

BEST_MODEL_PATH = 'best_model_navigator_10class.pth'
DATASET_DIR = 'dataset_navigator_10class'

import shutil
import os

# Cesta k datasetu
dataset_path = 'dataset_navigator_10class' 
checkpoint_path = os.path.join(dataset_path, '.ipynb_checkpoints')

# Smazání složky, pokud existuje
if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
    print("✅ Složka .ipynb_checkpoints byla úspěšně smazána!")
else:
    print("Složka .ipynb_checkpoints nebyla nalezena (už je to v pořádku).")

✅ Složka .ipynb_checkpoints byla úspěšně smazána!


In [2]:
# Definice transformací (náhodné změny barev pro lepší odolnost)
transforms_pipeline = transforms.Compose([
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Načtení dat
# ImageFolder automaticky přiřadí třídy podle názvů složek (0_..., 1_...)
dataset = datasets.ImageFolder(DATASET_DIR, transforms_pipeline)

print(f"Nalezeno {len(dataset)} obrázků.")
print("Třídy:", dataset.classes)

Nalezeno 2000 obrázků.
Třídy: ['corner_left', 'corner_right', 'corridor', 'dead_end', 'drift_left', 'drift_right', 'goal', 't_junction_lr', 't_junction_sl', 't_junction_sr']


In [3]:
# Rozdělíme data: Většina na trénování, 10 % (nebo fixních 50 ks) na testování
test_percent = 0.15 # 15% dat na testování
num_test = int(len(dataset) * test_percent)
num_train = len(dataset) - num_test

train_dataset, test_dataset = torch.utils.data.random_split(dataset, [num_train, num_test])

# Vytvoření DataLoaderů (posílají data do GPU po dávkách)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0 # Na Jetson Nano raději 0 nebo 1
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

print(f"Trénovací sada: {num_train} obr. | Testovací sada: {num_test} obr.")

Trénovací sada: 1700 obr. | Testovací sada: 300 obr.


In [4]:
# Stáhneme předtrénovaný ResNet18
model = models.resnet18(pretrained=True)

#10 vstupů
model.fc = torch.nn.Linear(512, 10)

# Přesun na grafickou kartu (GPU)
device = torch.device('cuda')
model = model.to(device)

print("Model připraven na GPU (10 výstupů).")

Model připraven na GPU (10 výstupů).


In [ ]:
NUM_EPOCHS = 50  # Počet cyklů (pro 8 tříd doporučuji alespoň 30-50)
best_accuracy = 0.0

optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

print("Začínám trénovat...")

for epoch in range(NUM_EPOCHS):
    # 1. Trénování
    model.train()
    for images, labels in iter(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        optimizer.step()
    
    # 2. Testování (Validace)
    model.eval()
    test_error_count = 0.0
    for images, labels in iter(test_loader):
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        # Porovnáme, zda se model trefil (argmax vrátí index s nejvyšší pravděpodobností)
        test_error_count += float(torch.sum(torch.abs(labels - outputs.argmax(1))))
    
    # Výpočet přesnosti
    test_accuracy = 1.0 - float(test_error_count) / float(len(test_dataset))
    
    print('Epoch %d: Přesnost: %f' % (epoch, test_accuracy))
    
    # Uložení, pokud je to zatím nejlepší výsledek
    if test_accuracy > best_accuracy:
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        best_accuracy = test_accuracy
        print(f" -> Model uložen! (Nová nejlepší přesnost: {best_accuracy:.2f})")

Začínám trénovat...
Epoch 0: Přesnost: 0.956667
 -> Model uložen! (Nová nejlepší přesnost: 0.96)
Epoch 1: Přesnost: 0.983333
 -> Model uložen! (Nová nejlepší přesnost: 0.98)
Epoch 2: Přesnost: 0.976667
Epoch 3: Přesnost: 0.970000
Epoch 4: Přesnost: 0.976667
Epoch 5: Přesnost: 0.983333
Epoch 6: Přesnost: 0.970000
Epoch 7: Přesnost: 0.983333
Epoch 8: Přesnost: 0.980000
Epoch 9: Přesnost: 0.983333
Epoch 10: Přesnost: 0.990000
 -> Model uložen! (Nová nejlepší přesnost: 0.99)
Epoch 11: Přesnost: 0.990000
Epoch 12: Přesnost: 0.990000
Epoch 13: Přesnost: 0.983333
Epoch 14: Přesnost: 0.983333
Epoch 15: Přesnost: 0.986667
Epoch 16: Přesnost: 0.986667
Epoch 17: Přesnost: 0.986667
Epoch 18: Přesnost: 0.990000
Epoch 19: Přesnost: 0.986667
Epoch 20: Přesnost: 0.986667
Epoch 21: Přesnost: 0.986667
Epoch 22: Přesnost: 0.990000
